# Cleaning and Filtering the Movies Dataset

In the second notebook, we conducted an Exploratory Data Analysis (EDA) on the movies dataset to ensure we have enough data and understand its structure. In this notebook, I will clean and filter some movies so we have only relevant films for our recommender system

In [27]:
import pandas as pd
import ast
import numpy as np
import warnings
warnings.filterwarnings("ignore")

## Movies Dataset

In [28]:
mdf = pd.read_parquet('../data/raw/movies_metadata.parquet')
mdf.columns

Index(['adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'genres',
       'homepage', 'id', 'imdb_id', 'origin_country', 'original_language',
       'original_title', 'overview', 'popularity', 'poster_path',
       'production_companies', 'production_countries', 'release_date',
       'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title',
       'video', 'vote_average', 'vote_count', 'movieId'],
      dtype='object')

In [29]:
# Select only the relevant columns
mdf = mdf[['movieId', 'id', 'title', 'genres', 'overview', 
           'release_date', 'runtime', 'tagline',  'vote_average', 
           'popularity', 'vote_count', 'poster_path', 'backdrop_path']]

mdf.head().transpose()

,0,1,2,3,4
movieId,8,7,5,9,3
id,45325,11860,11862,9091,15602
title,Tom and Huck,Sabrina,Father of the Bride Part II,Sudden Death,Grumpier Old Men
genres,"[{'id': 10751, 'name': 'Family'}, {'id': 28, '...","[{'id': 10749, 'name': 'Romance'}, {'id': 18, ...","[{'id': 35, 'name': 'Comedy'}, {'id': 10751, '...","[{'id': 28, 'name': 'Action'}, {'id': 18, 'nam...","[{'id': 10749, 'name': 'Romance'}, {'id': 35, ..."
overview,"A mischievous young boy, Tom Sawyer, witnesses...","Sabrina Fairchild, a chauffeur's daughter, gre...",Just when George Banks has recovered from his ...,When a man's daughter is suddenly taken during...,A family wedding reignites the ancient feud be...
release_date,1995-12-22,1995-12-15,1995-12-08,1995-10-27,1995-12-22
runtime,97,127,106,110,101
tagline,A lot of kids get into trouble. These two inve...,You are cordially invited to the most surprisi...,Just when his world is back to normal... he's ...,Terror goes into overtime.,Still Yelling. Still Fighting. Still Ready for...
vote_average,5.3,6.215,6.252,5.999,6.467
popularity,0.7729,2.038,1.6479,2.5257,2.9176


In [30]:
mdf.dtypes

movieId            int64
id                 int64
title             object
genres            object
overview          object
release_date      object
runtime            int64
tagline           object
vote_average     float64
popularity       float64
vote_count         int64
poster_path       object
backdrop_path     object
dtype: object

It seems that all the features are in its correct data type

In [31]:
print(f'The original movies dataset has {mdf.shape[0]:,} movies')

The original movies dataset has 86,279 movies


## Cleaning and Preprocessing  the Dataset

We will clean the dataset by removing rows that lack important information. Specifically, we will remove movies that do not have a `title`, `overview`, or `genre`, as these are essential for identifying movies and calculating similarities in a content-based recommender system. Without a title, we cannot determine which movie it is, and without an overview or genre, we cannot derive meaningful movie comparisons.

Also we will transform the data types of certain columns to ensure they are properly formatted for further analysis. Specifically, we will extract and convert the genres, collection (if available), and production companies, as these columns are currently stored as stringified dictionaries.


In [32]:
# Drop films without title or genre
mdf.dropna(subset='title', inplace=True)
mdf.dropna(subset='overview', inplace=True)

In [33]:
# Extract the genres
mdf['genres'] =  mdf['genres'].apply(lambda x: [item['name'] for item in x])
mdf.head().transpose()

,0,1,2,3,4
movieId,8,7,5,9,3
id,45325,11860,11862,9091,15602
title,Tom and Huck,Sabrina,Father of the Bride Part II,Sudden Death,Grumpier Old Men
genres,"[Family, Action, Adventure, Drama]","[Romance, Drama, Comedy]","[Comedy, Family]","[Action, Drama, Thriller, Crime]","[Romance, Comedy]"
overview,"A mischievous young boy, Tom Sawyer, witnesses...","Sabrina Fairchild, a chauffeur's daughter, gre...",Just when George Banks has recovered from his ...,When a man's daughter is suddenly taken during...,A family wedding reignites the ancient feud be...
release_date,1995-12-22,1995-12-15,1995-12-08,1995-10-27,1995-12-22
runtime,97,127,106,110,101
tagline,A lot of kids get into trouble. These two inve...,You are cordially invited to the most surprisi...,Just when his world is back to normal... he's ...,Terror goes into overtime.,Still Yelling. Still Fighting. Still Ready for...
vote_average,5.3,6.215,6.252,5.999,6.467
popularity,0.7729,2.038,1.6479,2.5257,2.9176


Delete the films of which we do not have their genres

In [34]:
mdf['genres'] = mdf['genres'].apply(lambda x: np.nan if not x else x)
mdf.dropna(subset='genres', inplace=True)

#### Poster Path

The poster path will be useful to display the poster of the movie in the user interface, as well as the backdrop poster; so let's check how many posters are missing

In [35]:
mdf['poster_path'].isnull().sum()

np.int64(1062)

In [36]:
mdf['backdrop_path'].isnull().sum()

np.int64(10426)

In [37]:
# Sort by popularity the movies that do not have a poster path
mdf[mdf['poster_path'].isnull()].sort_values(by='popularity', ascending=False)[['title', 'popularity']].head(10)

,title,popularity
76768,Hotel Room,6.7308
36741,Pian delle stelle,6.5315
81623,Miraculous Place,4.8176
53888,Trofim,4.5697
44538,Peng! Du bist tot!,4.3255
41299,Dogpound Shuffle,4.2402
82076,In der Sache I. Robert Oppenheimer,4.1450
43939,Bright Night,4.0651
64906,Будь со мной,3.8521
84527,Die große Woge,3.7775


In [38]:
mdf[mdf['backdrop_path'].isnull()].\
    sort_values(by='popularity', ascending=False)[['title', 'popularity', 'vote_average', 'vote_count']].head(10)

,title,popularity,vote_average,vote_count
16509,Girl Play,10.3995,5.700,29
51186,Pandiyan,7.7246,5.333,6
54488,Thillana Mohanambal,7.6277,7.100,9
79785,Torrid Noon,7.5355,6.333,6
65972,Angel Force,7.1297,6.500,5
38484,Das Stunden-Hotel von St. Pauli,7.0459,3.000,2
81875,Apoorvaragam,6.9632,6.600,7
53914,Captain Prabhakaran,6.8343,6.600,6
55350,Dhill,6.8296,5.300,18
51183,Mannan,6.8173,6.900,8


The movies without a poster path appear to be quite unpopular. Since the posters are essential for displaying the film in the final UI, we will remove the films that lack a poster path. Also notice that the films that lack of this information are not populars or have a lot of votes.

In [39]:
mdf.dropna(subset='poster_path', inplace=True)

In [40]:
mdf.dropna(subset='backdrop_path', inplace=True)

In [41]:
print(f'After doing some cleaning we are left with {mdf.shape[0]:,} movies')

After doing some cleaning we are left with 74,846 movies


## Filtering

To enhance the performance of our recommender system, we will filter and select only relevant movies for the following reasons:

- **Relevance to Modern Audiences:** Older movies may not resonate with today’s viewers. By focusing on more recent or popular titles, we ensure that recommendations remain aligned with current trends and user preferences.
- **Avoiding Data Sparsity:** Older movies typically have fewer interactions and ratings, leading to data sparsity. Since recommender systems rely on user interactions, movies with limited data may not generate meaningful recommendations.
- **Reducing Complexity:** A smaller, more focused dataset of relevant movies reduces model complexity. Working with fewer, more relevant movies means fewer features to process, resulting in faster computation and more efficient learning.
- **Improving User-Item Interactions:** Users are generally more engaged with recent or trending movies. Filtering out older titles helps focus the system on movies with more active user interactions, thereby enhancing the model’s accuracy.

Filtering movies by vote count, popularity, or average rating can often lead to misleading results. To address this, we apply IMDB's weighted rating formula, which balances a movie's average rating with the number of votes it has received. In other words, a high rating alone isn't enough — the movie also needs a significant number of votes to be considered trustworthy.

$$\text{WR} = \left( \frac{v}{v + m} \cdot R \right) + \left( \frac{m}{v + m} \cdot C \right)$$

where:

  - $v$ is the number of votes for the movie
  - $m$ is the minimum number of votes required for relevance
  - $R$ is the average rating of the movie
  - $C$ is the mean rating across all movies in the dataset

Since we have nearly 90,000 movies and 75% of them have 82 votes or fewer (which is relatively low), we will set $m$ to the 90th percentile of the vote count distribution. This means a movie must have more votes than 90\% of all movies in the dataset to be considered relevant.

In [42]:
m = mdf['vote_count'].quantile(0.9)
C = mdf['vote_average'].mean()

def weighted_rating(x, m=m, C=C):
    v = x['vote_count']
    R = x['vote_average']
    return (v / (v + m) * R) + (m / (v + m) * C)

In [43]:
mdf['score'] = mdf.apply(weighted_rating, axis=1)

# Top 5 movies according to IMDb's score
mdf.sort_values(by='score', ascending=False).head()

,movieId,id,title,genres,overview,release_date,runtime,tagline,vote_average,popularity,vote_count,poster_path,backdrop_path,score
313,318,278,The Shawshank Redemption,"[Drama, Crime]",Imprisoned in the 1940s for the double murder ...,1994-09-23,142,Fear can hold you prisoner. Hope can set you f...,8.712,25.9882,28785,/9cqNxx0GxF0bflZmeSMuL5tnGzr.jpg,/pNjh59JSxChQktamG3LMp9ZoQzp.jpg,8.669252
840,858,238,The Godfather,"[Drama, Crime]","Spanning the years 1945 to 1955, a chronicle o...",1972-03-14,175,An offer you can't refuse.,8.686,24.3675,21769,/3bhkrj58Vtu7enYsRolD1fZdja1.jpg,/htuuuEwAvDVECMpb0ltLLyZyDDt.jpg,8.630291
522,527,424,Schindler's List,"[Drama, History, War]",The true story of how businessman Oskar Schind...,1993-12-15,195,THE BOSS BABY IS BACK...AND HE MEANS BUSINESS!,8.565,12.6860,16662,/sF1U4EUQS8YHUYjNl3pMGNIQyr0.jpg,/zb6fM1CX41D9rF9hdgclu0peUmy.jpg,8.495893
12165,58559,155,The Dark Knight,"[Drama, Action, Crime, Thriller]",Batman raises the stakes in his war on crime. ...,2008-07-16,152,Welcome to a world without rules.,8.522,29.4224,34294,/qJ2tW6WMUDux911r6m7haRef0WH.jpg,/enNubozHn9pXi0ycTVYUWfpHZm.jpg,8.488522
1181,1221,240,The Godfather Part II,"[Drama, Crime]",In the continuing saga of the Corleone crime f...,1974-12-20,202,The rise and fall of the Corleone empire.,8.571,13.5960,13142,/hek3koDUyRQk7FIhPXsa6mT2Zc3.jpg,/kGzFbGhp99zva6oZODW5atUtnqi.jpg,8.483803


As we can see this filtering step was pretty good, since the top movies we can see are authentic gems.

### Year & Rating

We will extract the release year of the films and select movies released in 1990 or later, since our target audience is mostly yougn people, born in the 2000's.

In [44]:
mdf['release_date'] = pd.to_datetime(mdf['release_date'], errors='coerce')
mdf['year'] = mdf['release_date'].dt.year.fillna(1989).astype('int')
# Drop the release date
mdf = mdf.drop(columns=['release_date']).reset_index(drop=True)
# Filter the movies
mdf = mdf[ (mdf['year'] > 1994)]

Finally we will select the best 5,000 movies according to this score

In [45]:
mdf = mdf.sort_values(by='score', ascending=False).head(5000)

# Ensure the size is the correct
mdf.shape

(5000, 14)

In [46]:
# Top 10 movies with the highest IMDB Score
mdf.head(10)

,movieId,id,title,genres,overview,runtime,tagline,vote_average,popularity,vote_count,poster_path,backdrop_path,score,year
11779,58559,155,The Dark Knight,"[Drama, Action, Crime, Thriller]",Batman raises the stakes in his war on crime. ...,152,Welcome to a world without rules.,8.522,29.4224,34294,/qJ2tW6WMUDux911r6m7haRef0WH.jpg,/enNubozHn9pXi0ycTVYUWfpHZm.jpg,8.488522,2008
19856,109487,157336,Interstellar,"[Adventure, Drama, Science Fiction]",The adventures of a group of explorers who mak...,169,Mankind was born on Earth. It was never meant ...,8.500,42.3136,37721,/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg,/vgnoBSVzWAV9sNQUORaDGvDp7wx.jpg,8.469790,2014
5283,5618,129,Spirited Away,"[Animation, Family, Fantasy]","A young girl, Chihiro, becomes trapped in a st...",125,On the other side of the tunnel was a mysterio...,8.535,32.6674,17422,/39wmItIWsg5sZMyRUHLkWBcuVCM.jpg,/ukfI9QkU1aIhOhKXYWE9n3z1mFR.jpg,8.469596,2001
6749,7153,122,The Lord of the Rings: The Return of the King,"[Adventure, Fantasy, Action]",As armies mass for a final battle that will de...,201,There can be no triumph without loss. No victo...,8.488,21.3341,25392,/rCzpDGLbOoPwLjy3OAm5NUPOTrC.jpg,/2u7zbn8EudG6kLlBzUYqP8RyFU4.jpg,8.443592,2003
2907,3147,497,The Green Mile,"[Fantasy, Drama, Crime]",A supernatural tale set on death row in a Sout...,189,Paul Edgecomb didn't believe in miracles. Unti...,8.503,14.0263,18318,/8VG8fDNiy50H4FedGwdSVUPoaJe.jpg,/vxJ08SvwomfKbpboCWynC3uqUg4.jpg,8.441495,1999
53265,202439,496243,Parasite,"[Comedy, Thriller, Drama]","All unemployed, Ki-taek's family takes peculia...",133,Act like you own the place.,8.498,17.2124,19407,/7IiTTgloJzvGI1TAYymCfbfl3vT.jpg,/hiKmpZMGZsrkA3cdce8a7Dpos1j.jpg,8.439982,2019
2726,2959,550,Fight Club,"[Drama, Thriller]",A ticking-time-bomb insomniac and a slippery s...,139,Mischief. Mayhem. Soap.,8.437,20.4222,30662,/jSziioSwPVrOy9Yow3XhWIBDjq1.jpg,/hZkgoQYus5vegHoetLkCJzb17zJ.jpg,8.400860,1999
37959,163134,372058,Your Name.,"[Animation, Romance, Drama]",High schoolers Mitsuha and Taki are complete s...,106,"Separated by distance, connected by fate.",8.481,15.6380,11940,/q719jXXEzOoYaps6babgKnONONX.jpg,/8x9iKH8kWA0zdkgNdpAew7OstYe.jpg,8.388659,2016
4679,4993,120,The Lord of the Rings: The Fellowship of the Ring,"[Adventure, Fantasy, Action]","Young hobbit Frodo Baggins, after inheriting a...",179,One ring to rule them all.,8.424,24.5972,26308,/6oom5QYQ2yQTMJIbnvbkBL9cHo6.jpg,/a0lfia8tk8ifkrve0Tn8wkISUvs.jpg,8.382202,2001
5598,5952,121,The Lord of the Rings: The Two Towers,"[Adventure, Fantasy, Action]",Frodo Baggins and the other members of the Fel...,179,The journey continues.,8.409,17.4150,22850,/5VTN0pR8gcqV3EPUHHfMGnJYN9L.jpg,/mshaKLtPUxcDBhzau6qiObEblhL.jpg,8.361294,2002


As we can see, the movies we've kept are pretty popular, which should really help the recommender system improve the quality of its suggestions. I decided to keep 5,000 movies out of the original 87,500 because I’m working on this project locally and don’t have the resources to handle millions of ratings and tens of thousands of movies.

In [47]:
# Save only the ids on a csv file since we already have its metadata in another file, so we do not duplicated this data
mdf[['movieId', 'id']].to_csv('../data/processed/clean_movies_ids.csv', index=False)